In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex
import citation_utils
import metric_utils
import reranker_utils
import rrf
import hits_utils


libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
# BGE-embedding

from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS
/root/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4

DenseIndex.embeddings:  (2107648, 1024)


In [16]:
%load_ext autoreload

%autoreload 2

from pipeline import Pipeline
import math

p = Pipeline(court_consideration_df,
             court_consideration_d,
             law_df,
             law_d,
             dense_index,
             sparse_index,
             reranker,
             test_df,
             valid_df,
             citation_agg_w1=0.1,
             citation_agg_w2=0.3,
             citation_agg_w3=0.6,
             global_citaion_ranking_pool_method='sum',
             global_citation_ranking_agg_weight=0.7,
             global_citation_ranking_recall_weight=0.3,
             global_citation_ranking_recall_decay_fn=lambda x: math.sqrt(x),
             false_positive_threshold_score = 1.0,
             window_size=5,
             step=3
            )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
p.evaluate(stop=None)

evaluation:   0%|          | 0/10 [00:00<?, ?it/s]

[recall] hit_with_score_l.len: 1612
[normalize_sr] count: 34
[rerank] sliced: 1232/7980
agg [('Art. 221 Abs. 1 lit', 9.153023021015905), ('Art. 212 Abs. 2 lit', 8.906489421202329), ('Art. 5 Ziff', 5.199339760512634), ('BGE 137 IV 84 E. 3', 3.2462620074975646), ('Art. 221 StPO', 3.026403838670462)]
recall [('Art. 221 Abs. 1 lit', 9.5401193844025), ('Art. 221 Abs. 1 StPO', 9.063332565474072), ('Art. 221 StPO', 8.827049851565995), ('Art. 212 Abs. 2 lit', 8.128362341396166), ('Art. 237 ff', 6.380422940959132)]
[recall] hit_with_score_l.len: 1860
[normalize_sr] count: 37
[rerank] sliced: 1422/12665
agg [('Art. 16 ATSG', 3.2055797855932684), ('BGE 132 V 393 E. 3', 3.0564142979154956), ('Art. 61 lit', 2.8824499511065924), ('Art. 6 ATSG', 2.8808751154097783), ('Art. 8 Abs. 1 ATSG', 2.559432967957896)]
recall [('Art. 8 Abs. 1 ATSG', 7.680298944853919), ('art. 8 LPGA', 6.033674790360355), ('art. 16 LPGA', 6.030276837676167), ('art. 18 al', 5.4978808742146885), ('art. 6 al', 5.142667324105352)]
[

In [ ]:
sub_df = p.generate_submission(50)
sub_df.to_csv("../output/submission_50.csv", index=False)

In [ ]:
limit=40
sub_df = pd.read_csv("../output/submission_50.csv")
pred_l = []
for query_id, pred in zip(sub_df['query_id'].tolist(), sub_df['predicted_citations'].tolist()):
    pred2 = pred.split(";")[:limit]
    pred_l.append(';'.join(pred2))
new_sub_df = pd.DataFrame({'query_id':sub_df['query_id'].tolist(), 'predicted_citations':pred_l})
new_sub_df.to_csv("../output/submission.csv", index=False)